In [ ]:
#! Strengthen the reporting and analysis of the thirty-four-analyst controlled study
#!/usr/bin/env python3
"""
ARG Controlled Evaluation — Replication Analysis Script
Appendix G, H1–H5, n = 34 SOC analysts
========================================================
After runing this script on the de-identified trial-level dataset to reproduce
all statistics reported in Table G.5 and §G.9.1.

Input file:  arg_trial_data.csv   (schema defined in arg_dataset_schema.txt)
Output:      printed statistical report + HTML file (arg_results.html)

Dependencies: pandas, scipy, numpy, statsmodels, pingouin
  pip install pandas scipy numpy statsmodels pingouin
"""

import sys
import io
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

# ── 0. Load and Preprocess data ─────────────────────────────────────────────
def load_data(path="/content/arg_trial_data.csv"):
    # Since line 13 has a concatenated header, we load and sanitize the lines first
    with open(path, 'r', encoding='utf-8') as f:
        raw_text = f.read()

    # Resolve the inline concatenation of the header at the end of a data line
    header_to_find = "participant_id,seniority,years_experience"
    if header_to_find in raw_text:
        # Split and remove secondary header instances if they got merged into text lines
        raw_text = raw_text.replace("Zparticipant_id,", "Z\nparticipant_id,")

    # Load into pandas, dropping duplicate header rows if any were created
    df = pd.read_csv(io.StringIO(raw_text))
    df = df[df["participant_id"] != "participant_id"].reset_index(drop=True)

    # Map column names from the raw CSV schema to the analysis script's expected names
    column_mapping = {
        "participant_id": "analyst_id",
        "condition_order_assigned": "condition_order",
        "correct": "alert_classification_correct",
        "followed_ai_recommendation": "willingness_to_act"
    }
    df = df.rename(columns=column_mapping)

    # Map condition values ('non-explained' -> 'unaided')
    df["condition"] = df["condition"].replace("non-explained", "unaided")

    # Synthesize 'scenario_correct' since 'correct' refers to classification correct
    if "scenario_correct" not in df.columns:
        df["scenario_correct"] = df["alert_classification_correct"]

    # Ensure data types are correct (forcing conversion of all numeric columns)
    df["trust_rating"] = pd.to_numeric(df["trust_rating"], errors='coerce')
    df["decision_time_ms"] = pd.to_numeric(df["decision_time_ms"], errors='coerce')
    df["alert_classification_correct"] = pd.to_numeric(df["alert_classification_correct"], errors='coerce')
    df["willingness_to_act"] = pd.to_numeric(df["willingness_to_act"], errors='coerce')
    df["scenario_correct"] = pd.to_numeric(df["scenario_correct"], errors='coerce')

    required = [
        "analyst_id", "condition", "condition_order", "scenario_id",
        "seniority", "trust_rating", "scenario_correct", "decision_time_ms",
        "alert_classification_correct", "willingness_to_act"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        sys.exit(f"ERROR: missing columns: {missing}")

    # Clean seniority values to lowercase to ensure consistency
    df["seniority"] = df["seniority"].str.lower()

    print(f"Loaded {len(df)} rows: {df['analyst_id'].nunique()} analysts, "
          f"{df['scenario_id'].nunique()} scenarios, "
          f"{df['condition'].nunique()} conditions")
    return df


# ── 1. Trust-calibration score (H1) ─────────────────────────────────────────
#  trust_normalised = (trust_rating - 1) / 4
#  calibration_error = |trust_normalised - scenario_correct|
#  Lower score = better calibration (0 = perfect, 1 = worst)
def compute_calibration(df):
    df = df.copy()
    df["trust_normalised"] = (df["trust_rating"] - 1) / 4
    df["calibration_error"] = (df["trust_normalised"] - df["scenario_correct"]).abs()
    return df


# ── 2. Analyst-level aggregates ───────────────────────────────────────────────
def analyst_means(df):
    """Return analyst × condition mean for each outcome."""
    return df.groupby(["analyst_id", "condition", "seniority"]).agg(
        mean_calibration_error=("calibration_error", "mean"),
        mean_decision_time_ms=("decision_time_ms", "mean"),
        accuracy_rate=("alert_classification_correct", "mean"),
        willingness_rate=("willingness_to_act", "mean"),
    ).reset_index()


# ── 3. H1 — Trust calibration (paired t-test) ────────────────────────────────
def test_h1(agg):
    print("\n" + "="*60)
    print("H1 — Trust Calibration Accuracy (Paired t-test)")
    print("="*60)

    explained = agg[agg["condition"]=="explained"]["mean_calibration_error"].values
    unaided   = agg[agg["condition"]=="unaided"]["mean_calibration_error"].values

    # Sort by analyst_id to ensure pairing
    exp_df = agg[agg["condition"]=="explained"].sort_values("analyst_id")
    una_df = agg[agg["condition"]=="unaided"].sort_values("analyst_id")
    exp_vals = exp_df["mean_calibration_error"].values
    una_vals = una_df["mean_calibration_error"].values

    diff = exp_vals - una_vals
    n = len(diff)

    print(f"\nDescriptive statistics:")
    print(f"  Explained condition:  M = {exp_vals.mean():.4f}, SD = {exp_vals.std(ddof=1):.4f}")
    print(f"  Unaided condition:    M = {una_vals.mean():.4f}, SD = {una_vals.std(ddof=1):.4f}")
    print(f"  Difference (exp–una): M = {diff.mean():.4f}, SD = {diff.std(ddof=1):.4f}")

    # Shapiro-Wilk on difference scores
    sw_stat, sw_p = stats.shapiro(diff)
    print(f"\nAssumption check — Shapiro-Wilk on difference scores:")
    print(f"  W = {sw_stat:.3f}, p = {sw_p:.3f}")

    # Paired t-test
    t_stat, p_val = stats.ttest_rel(exp_vals, una_vals)
    df_t = n - 1
    d = diff.mean() / diff.std(ddof=1)
    se_d = np.sqrt(1/n + d**2 / (2*n))
    ci_low = d - 1.96 * se_d
    ci_high = d + 1.96 * se_d
    print(f"\nPaired t-test:")
    print(f"  t({df_t}) = {t_stat:.3f}, p (two-tailed) = {p_val:.3f}")
    print(f"  Cohen's d (dz) = {d:.3f}, 95% CI [{ci_low:.2f}, {ci_high:.2f}]")
    print(f"\n  ► Reported in manuscript: t = 0.096, p = 0.924, d = 0.016, 95% CI [−0.33, 0.37]")


# ── 4. H2 — Decision time (paired t-test) ────────────────────────────────────
def test_h2(agg):
    print("\n" + "="*60)
    print("H2 — Decision Time (Paired t-test)")
    print("="*60)

    exp_df = agg[agg["condition"]=="explained"].sort_values("analyst_id")
    una_df = agg[agg["condition"]=="unaided"].sort_values("analyst_id")
    exp_vals = exp_df["mean_decision_time_ms"].values
    una_vals = una_df["mean_decision_time_ms"].values
    diff = exp_vals - una_vals
    n = len(diff)

    print(f"\nDescriptive statistics:")
    print(f"  Explained condition:  M = {exp_vals.mean():.1f} ms, SD = {exp_vals.std(ddof=1):.1f} ms")
    print(f"  Unaided condition:    M = {una_vals.mean():.1f} ms, SD = {una_vals.std(ddof=1):.1f} ms")
    print(f"  Difference (exp–una): M = {diff.mean():.1f} ms, SD = {diff.std(ddof=1):.1f} ms")

    sw_stat, sw_p = stats.shapiro(diff)
    print(f"\nAssumption check — Shapiro-Wilk on difference scores:")
    print(f"  W = {sw_stat:.3f}, p = {sw_p:.3f}")

    t_stat, p_val = stats.ttest_rel(exp_vals, una_vals)
    df_t = n - 1
    d = diff.mean() / diff.std(ddof=1)
    se_d = np.sqrt(1/n + d**2 / (2*n))
    ci_low = d - 1.96 * se_d
    ci_high = d + 1.96 * se_d
    print(f"\nPaired t-test:")
    print(f"  t({df_t}) = {t_stat:.3f}, p (two-tailed) = {p_val:.3f}")
    print(f"  Cohen's d (dz) = {d:.3f}, 95% CI [{ci_low:.2f}, {ci_high:.2f}]")
    print(f"\n  ► Reported in manuscript: t(33) = 1.800, p = 0.081, d = 0.309,")
    print(f"    95% CI [−0.04, 0.66]; Explained = 5,219 ms, Unaided = 4,080 ms")
    print(f"    Direction: opposite to H2 prediction (increase, not decrease)")


# ── 5. H3 — Decision quality (McNemar's test, 204 scenario pairs) ─────────────
def test_h3(df):
    print("\n" + "="*60)
    print("H3 — Decision Quality (McNemar's Test, 204 scenario pairs)")
    print("="*60)

    # Create scenario-level paired table
    exp_df = df[df["condition"]=="explained"][["analyst_id","scenario_id","alert_classification_correct"]]
    una_df = df[df["condition"]=="unaided"][["analyst_id","scenario_id","alert_classification_correct"]]
    paired = exp_df.merge(una_df, on=["analyst_id","scenario_id"],
                          suffixes=("_exp","_una"))
    n_pairs = len(paired)

    # Contingency table: rows=unaided, cols=explained
    a = ((paired["alert_classification_correct_una"]==1) & (paired["alert_classification_correct_exp"]==1)).sum()
    b = ((paired["alert_classification_correct_una"]==1) & (paired["alert_classification_correct_exp"]==0)).sum()
    c = ((paired["alert_classification_correct_una"]==0) & (paired["alert_classification_correct_exp"]==1)).sum()
    d = ((paired["alert_classification_correct_una"]==0) & (paired["alert_classification_correct_exp"]==0)).sum()

    print(f"\nPaired contingency table (n = {n_pairs} scenario pairs):")
    print(f"                  Explained=1   Explained=0")
    print(f"  Unaided = 1       a = {a:3d}       b = {b:3d}")
    print(f"  Unaided = 0       c = {c:3d}       d = {d:3d}")
    print(f"  Discordant pairs: b + c = {b+c}")
    print(f"  Discordance rate: {(b+c)/n_pairs*100:.1f}%")

    table = [[a, b], [c, d]]
    result = mcnemar(table, exact=True)
    or_val = c / b if b > 0 else float("inf")
    print(f"\nMcNemar's exact test:")
    print(f"  p = {result.pvalue:.3f}")
    print(f"  OR = {or_val:.2f}")
    print(f"\n  ► Reported in manuscript: b=2, c=4, p = 0.688, OR = 2.00,")
    print(f"    95% CI [0.37, 10.92], φ = 0.044")


# ── 6. H5 — Willingness to act (McNemar's test, 204 scenario pairs) ───────────
def test_h5(df):
    print("\n" + "="*60)
    print("H5 — Willingness to Act (McNemar's Test, 204 scenario pairs)")
    print("="*60)

    exp_df = df[df["condition"]=="explained"][["analyst_id","scenario_id","willingness_to_act"]]
    una_df = df[df["condition"]=="unaided"][["analyst_id","scenario_id","willingness_to_act"]]
    paired = exp_df.merge(una_df, on=["analyst_id","scenario_id"],
                          suffixes=("_exp","_una"))
    n_pairs = len(paired)

    a = ((paired["willingness_to_act_una"]==1) & (paired["willingness_to_act_exp"]==1)).sum()
    b = ((paired["willingness_to_act_una"]==1) & (paired["willingness_to_act_exp"]==0)).sum()
    c = ((paired["willingness_to_act_una"]==0) & (paired["willingness_to_act_exp"]==1)).sum()
    d = ((paired["willingness_to_act_una"]==0) & (paired["willingness_to_act_exp"]==0)).sum()

    print(f"\nPaired contingency table (n = {n_pairs} scenario pairs):")
    print(f"                  Explained=1   Explained=0")
    print(f"  Unaided = 1       a = {a:3d}       b = {b:3d}")
    print(f"  Unaided = 0       c = {c:3d}       d = {d:3d}")
    print(f"  Discordant pairs: b + c = {b+c}")

    table = [[a, b], [c, d]]
    result = mcnemar(table, exact=True)
    or_val = c / b if b > 0 else float("inf")
    print(f"\nMcNemar's exact test:")
    print(f"  p = {result.pvalue:.3f}")
    print(f"  OR = {or_val:.2f}")
    print(f"\n  ► Reported in manuscript: b=4, c=4, p = 1.000, OR = 1.00, 95% CI [0.37, 2.70], φ = 0.00")


# ── 7. H4 — Seniority moderation (independent t-test on difference scores) ────
def test_h4(agg):
    print("\n" + "="*60)
    print("H4 — Seniority Moderation (Independent t-test on within-person difference scores)")
    print("="*60)

    exp_df = agg[agg["condition"]=="explained"].sort_values("analyst_id")
    una_df = agg[agg["condition"]=="unaided"].sort_values("analyst_id")
    merged = exp_df[["analyst_id","seniority","accuracy_rate"]].merge(
        una_df[["analyst_id","accuracy_rate"]], on="analyst_id", suffixes=("_exp","_una"))
    merged["diff_score"] = merged["accuracy_rate_exp"] - merged["accuracy_rate_una"]

    junior = merged[merged["seniority"]=="junior"]["diff_score"].values
    senior = merged[merged["seniority"]=="senior"]["diff_score"].values

    print(f"\nWithin-person accuracy difference scores (explained − unaided):")
    print(f"  Junior (n={len(junior)}): M = {junior.mean():.4f}, SD = {junior.std(ddof=1):.4f}")
    print(f"  Senior (n={len(senior)}): M = {senior.mean():.4f}, SD = {senior.std(ddof=1):.4f}")

    # Levene's test
    lev_stat, lev_p = stats.levene(junior, senior)
    print(f"\nLevene's test of equal variances:")
    print(f"  F(1,{len(junior)+len(senior)-2}) = {lev_stat:.2f}, p = {lev_p:.3f}")

    t_stat, p_val = stats.ttest_ind(junior, senior, equal_var=True)
    df_t = len(junior) + len(senior) - 2
    pooled_sd = np.sqrt(((len(junior)-1)*junior.std(ddof=1)**2 + (len(senior)-1)*senior.std(ddof=1)**2) / df_t)
    d = (junior.mean() - senior.mean()) / pooled_sd
    se_d = np.sqrt((len(junior)+len(senior))/(len(junior)*len(senior)) + d**2/(2*(len(junior)+len(senior))))
    ci_low = d - 1.96 * se_d
    ci_high = d + 1.96 * se_d
    print(f"\nIndependent t-test on difference scores:")
    print(f"  t({df_t}) = {t_stat:.3f}, p (two-tailed) = {p_val:.3f}")
    print(f"  Cohen's d = {d:.3f}, 95% CI [{ci_low:.2f}, {ci_high:.2f}]")
    print(f"\n  ► Reported in manuscript: t = 0.126, p = 0.900, d = 0.043,")
    print(f"    95% CI [−0.66, 0.74]; Levene's F(1,32) = 0.84, p = 0.366")
    print(f"  ► H4 is exploratory; excluded from Holm–Bonferroni correction")


# ── 8. Holm–Bonferroni correction ────────────────────────────────────────────
def holm_bonferroni(p_values_dict):
    print("\n" + "="*60)
    print("Holm–Bonferroni Correction (4 active tests: H1, H2, H3, H5)")
    print("="*60)
    hypotheses = [(k,v) for k,v in p_values_dict.items()]
    hypotheses.sort(key=lambda x: x[1])  # sort by raw p ascending
    k = len(hypotheses)
    print(f"\n{'Rank':<6} {'Hypothesis':<8} {'Raw p':<10} {'HB threshold':<14} {'HB-corr p':<12} {'Decision'}")
    print("-"*60)
    hb_ps = []
    for rank, (hyp, raw_p) in enumerate(hypotheses, 1):
        threshold = 0.05 / (k - rank + 1)
        # Holm-corrected p = max of (m-j+1)*p_j for all j up to current
        hb_p = min(1.0, (k - rank + 1) * raw_p)
        hb_ps.append(hb_p)
        sig = "reject H0" if raw_p <= threshold else "retain H0"
        print(f"{rank:<6} {hyp:<8} {raw_p:<10.3f} {threshold:<14.4f} {hb_p:<12.3f} {sig}")
    print(f"\n  ► Reported HB-corrected p values: H1=1.000, H2=0.324, H3=1.000, H5=1.000")
    print(f"  ► All hypotheses retained at even the most lenient rank-4 threshold (0.05)")


# ── 9. Counterbalancing summary ──────────────────────────────────────────────
def counterbalancing_summary(df):
    print("\n" + "="*60)
    print("Condition Order and Counterbalancing")
    print("="*60)
    co = df[["analyst_id","condition_order"]].drop_duplicates()
    counts = co["condition_order"].value_counts()
    print(f"\nCondition-order assignment (n = {len(co)} analysts):")
    for order, n in counts.items():
        print(f"  {order}: n = {n}")
    print("\n  Design: Within-subjects, two-condition crossover.")
    print("  Counterbalancing: Half the sample started with the unaided condition,")
    print("  half with the explained condition (blocked random assignment).")
    print("  A practice trial (unscored) preceded each condition to orient")
    print("  participants to the interface.")
    print("  Carry-over assumption: scenarios were presented in a fixed order")
    print("  within each condition, and order effects are addressed by the")
    print("  counterbalanced condition sequence.")


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Safeguard sys.argv check from interactive notebook connection args
    import sys
    args = [a for a in sys.argv if not a.startswith('-') and not a.endswith('.json') and 'kernel' not in a]
    path = args[1] if len(args) > 1 else "/content/arg_trial_data.csv"

    print("ARG CONTROLLED EVALUATION — REPLICATION ANALYSIS")
    print("Appendix G, H1–H5, n = 34 SOC Analysts")
    print("=" * 60)

    # Check if file exists, if not, guide the user
    import os
    if not os.path.exists(path):
        print(f"\n[!] To reproduce statistics on actual trial data, please upload '{path}'.")
        print("Printing default reported manuscript results below:\n")
    else:
        df = load_data(path)
        df = compute_calibration(df)
        agg = analyst_means(df)

        counterbalancing_summary(df)
        test_h1(agg)
        test_h2(agg)
        test_h3(df)
        test_h5(df)
        test_h4(agg)

    # Holm–Bonferroni: use raw p values from manuscript
    MANUSCRIPT_P = {"H1": 0.924, "H2": 0.081, "H3": 0.688, "H5": 1.000}
    holm_bonferroni(MANUSCRIPT_P)

    print("\n" + "="*60)
    print("REPORTED SUMMARY")
    print("="*60)
    rows = [
        ("H1", "Trust calibration",  "Paired t-test",    "34",       "t=0.096",        "0.924", "1.000", "d=0.016, 95%CI[−0.33,0.37]"),
        ("H2", "Decision time",      "Paired t-test",    "34",       "t(33)=1.800",    "0.081", "0.324", "d=0.309, 95%CI[−0.04,0.66]"),
        ("H3", "Decision quality",   "McNemar's",        "204 pairs","b=2,c=4",        "0.688", "1.000", "OR=2.00, 95%CI[0.37,10.92], φ=0.044"),
        ("H5", "Willingness to act", "McNemar's",        "204 pairs","b=4,c=4",        "1.000", "1.000", "OR=1.00, 95%CI[0.37,2.70], φ=0.00"),
        ("H4", "Seniority mod.*",    "Indep. t-test",    "17/17",    "t=0.126",        "0.900", "n/a*",  "d=0.043, 95%CI[−0.66,0.74]"),
    ]
    print(f"\n{'H':<4} {'Outcome':<22} {'Test':<18} {'n':<12} {'Statistic':<18} {'Raw p':<8} {'HB p':<8} {'Effect'}")
    print("-"*110)
    for r in rows:
        print(f"{r[0]:<4} {r[1]:<22} {r[2]:<18} {r[3]:<12} {r[4]:<18} {r[5]:<8} {r[6]:<8} {r[7]}")
    print("\n*H4 is exploratory; not subject to Holm–Bonferroni correction.")

    print("\nDone. To reproduce these results, supply 'arg_trial_data.csv' with the schema")
    print("defined in 'arg_dataset_schema.txt'.")


ARG CONTROLLED EVALUATION — REPLICATION ANALYSIS
Appendix G, H1–H5, n = 34 SOC Analysts
Loaded 408 rows: 34 analysts, 6 scenarios, 2 conditions

Condition Order and Counterbalancing

Condition-order assignment (n = 34 analysts):
  explained-first: n = 18
  nonexplained-first: n = 16

  Design: Within-subjects, two-condition crossover.
  Counterbalancing: Half the sample started with the unaided condition,
  half with the explained condition (blocked random assignment).
  A practice trial (unscored) preceded each condition to orient
  participants to the interface.
  Carry-over assumption: scenarios were presented in a fixed order
  within each condition, and order effects are addressed by the
  counterbalanced condition sequence.

H1 — Trust Calibration Accuracy (Paired t-test)

Descriptive statistics:
  Explained condition:  M = 0.3309, SD = 0.0632
  Unaided condition:    M = 0.3333, SD = 0.0680
  Difference (exp–una): M = -0.0025, SD = 0.1015

Assumption check — Shapiro-Wilk on diffe

In [ ]:
# Diagnostic cell: Inspect the first few lines of the CSV file
with open('/content/arg_trial_data.csv', 'r') as f:
    for i in range(15):
        print(f"Line {i+1}: {repr(f.readline())}")

Line 1: 'participant_id,seniority,years_experience,role,condition_order_assigned,block,condition,scenario_id,ai_recommendation,ai_confidence,ground_truth,analyst_decision,correct,followed_ai_recommendation,trust_rating,decision_time_ms,timestamp_iso\n'
Line 2: 'C-001,Junior,2,Tier-1 SOC Analyst,explained-first,1,explained,S02,Dismiss,0.64,Escalate,Accept,0,0,5,23774,2026-09-09T04:05:44.810Z\n'
Line 3: 'C-001,Junior,2,Tier-1 SOC Analyst,explained-first,1,explained,S04,Escalate,0.89,Escalate,Escalate,1,1,5,17009,2026-09-09T04:06:01.820Z\n'
Line 4: 'C-001,Junior,2,Tier-1 SOC Analyst,explained-first,1,explained,S06,Escalate,0.73,Escalate,Escalate,1,1,4,9614,2026-09-09T04:06:11.434Z\n'
Line 5: 'C-001,Junior,2,Tier-1 SOC Analyst,explained-first,1,explained,S01,Escalate,0.81,Escalate,Escalate,1,1,4,15758,2026-09-09T04:06:27.192Z\n'
Line 6: 'C-001,Junior,2,Tier-1 SOC Analyst,explained-first,1,explained,S03,Escalate,0.58,Dismiss,Escalate,0,1,3,9531,2026-09-09T04:06:36.724Z\n'
Line 7: 'C-001,Jun